[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/experimental-psychology/blob/main/notebooks/survey_analysis.ipynb)

# Statistical Building Blocks: Survey Data Analysis

In this notebook you will learn to build your own statistical tests from scratch using simulation. Instead of relying on pre-packaged functions that assume specific distributions, we will construct tests by:

1. **Choosing a test statistic** (e.g., a difference in means, a correlation coefficient)
2. **Defining a null hypothesis** (e.g., "there is no relationship between these variables")
3. **Simulating** what the test statistic would look like under the null hypothesis
4. **Comparing** our actual observed statistic to the simulated null distribution

This approach -- often called a *permutation test* or *randomization test* -- is powerful because it works for **any** test statistic, not just the ones that have named formulas. By the end of this notebook, you will see that the results closely match the classical statistical tests, while giving you a much deeper understanding of what those tests are actually doing.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set plot style for cleaner visuals
sns.set_theme(style='whitegrid')
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

## Load and explore the data

We will load the class survey responses directly from the published Google Sheet. The column headers are the full question text from the survey, so we rename them to short, code-friendly names.

In [ ]:
# Load data from the Google Sheets export URL
data_url = 'https://docs.google.com/spreadsheets/d/1MvZoEIU5OdAOVtUTw8QoYQdoz1HcOhInSqeZcE0wQgg/export?format=csv'
df = pd.read_csv(data_url)

# Mapping from full question text to short column names
column_mapping = {
    'Timestamp': 'timestamp',
    'How many hours of sleep do you typically get per night?': 'sleep_hours',
    'On a scale of 1-10, how stressed do you generally feel?': 'stress_level',
    'On a scale of 1-10, how happy do you generally feel?': 'happiness',
    'How many hours per day do you spend on screens (outside of schoolwork)?': 'screen_time',
    'How many days per week do you exercise?': 'exercise_frequency',
    'How many caffeinated beverages do you consume per day?': 'caffeine_intake',
    'How many hours per week do you spend studying (outside of class)?': 'study_hours',
    'On a scale of 1-10, how socially active are you?': 'social_activity',
}

# Apply renaming with fuzzy matching: try exact match first, then substring match
renamed = {}
for old_col in df.columns:
    matched = False
    # Try exact match first
    if old_col in column_mapping:
        renamed[old_col] = column_mapping[old_col]
        matched = True
    else:
        # Try substring match
        for pattern, new_name in column_mapping.items():
            if pattern.lower() in old_col.lower() or old_col.lower() in pattern.lower():
                renamed[old_col] = new_name
                matched = True
                break
    if not matched:
        renamed[old_col] = old_col.lower().replace(' ', '_')

df = df.rename(columns=renamed)

# Drop the timestamp column (not needed for analysis)
if 'timestamp' in df.columns:
    df = df.drop(columns=['timestamp'])

print(f'Loaded {df.shape[0]} responses with {df.shape[1]} columns.')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Basic EDA: summary statistics
print('=== Summary Statistics ===')
display(df.describe())

# Correlation heatmap
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 8))
correlation_matrix = numeric_df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Correlation Heatmap of Survey Variables')
plt.tight_layout()
plt.show()

## Building your own statistical test

The recipe for a simulation-based statistical test is:

1. **Pick a test statistic.** This is a single number that captures the pattern you are interested in (e.g., a difference in means between two groups, or a correlation between two variables).

2. **Define the null hypothesis.** This is the boring explanation -- usually "there is no real effect" or "these variables are unrelated." Under the null hypothesis, any apparent pattern in the data is just due to chance.

3. **Simulate the null distribution.** We generate thousands of fake datasets that look like what we would expect if the null hypothesis were true. For each fake dataset, we compute our test statistic. This gives us a distribution of test statistic values under the null.

4. **Compare the actual data to the simulations.** We compute our test statistic on the real data and see where it falls in the null distribution. If the actual value is far out in the tails (i.e., very unlikely under the null), we have evidence against the null hypothesis.

The **p-value** is simply the proportion of simulated values that are as extreme as (or more extreme than) the actual observed value. A small p-value means the observed pattern would be unlikely to arise by chance alone.

In [ ]:
# Simulation-based test: Do high-sleepers (>= 7 hrs) have lower stress?

# Step 1: Split into two groups
high_sleep = df[df['sleep_hours'] >= 7]['stress_level'].dropna()
low_sleep = df[df['sleep_hours'] < 7]['stress_level'].dropna()

print(f'High sleep group (>= 7 hrs): n = {len(high_sleep)}, mean stress = {high_sleep.mean():.2f}')
print(f'Low sleep group (< 7 hrs):   n = {len(low_sleep)}, mean stress = {low_sleep.mean():.2f}')

# Step 2: Compute the actual test statistic (difference in means)
actual_diff = high_sleep.mean() - low_sleep.mean()
print(f'\nActual difference in means (high - low): {actual_diff:.3f}')

# Step 3: Permutation test -- shuffle group labels 10,000 times
n_permutations = 10_000
combined = np.concatenate([high_sleep.values, low_sleep.values])
n_high = len(high_sleep)

null_diffs = np.empty(n_permutations)
for i in range(n_permutations):
    shuffled = np.random.permutation(combined)
    null_diffs[i] = shuffled[:n_high].mean() - shuffled[n_high:].mean()

# Step 4: Compute p-value (two-sided)
p_value_sim = np.mean(np.abs(null_diffs) >= np.abs(actual_diff))
print(f'Simulation-based p-value (two-sided): {p_value_sim:.4f}')

# Plot the null distribution
plt.figure(figsize=(10, 5))
plt.hist(null_diffs, bins=50, density=True, alpha=0.7, color='steelblue',
         edgecolor='white', label='Null distribution')
plt.axvline(actual_diff, color='red', linewidth=2, linestyle='--',
            label=f'Actual difference = {actual_diff:.3f}')
plt.axvline(-actual_diff, color='red', linewidth=2, linestyle='--', alpha=0.5)
plt.xlabel('Difference in mean stress (high sleep - low sleep)')
plt.ylabel('Density')
plt.title('Permutation Test: Sleep and Stress')
plt.legend()
plt.tight_layout()
plt.show()

## Try it yourself

The permutation test above can be adapted to test **any** hypothesis about group differences. You can also test *correlations* using a similar simulation approach: shuffle one variable to break the relationship, then see how extreme the actual correlation is compared to the shuffled versions.

Below is a template for testing whether two continuous variables are correlated. Try modifying the variable names to explore your own hypotheses!

In [ ]:
# Simulation-based correlation test
# Change these variable names to test your own hypothesis!
var_x = 'screen_time'
var_y = 'happiness'

# Clean data: drop rows where either variable is missing
clean = df[[var_x, var_y]].dropna()
x_vals = clean[var_x].values
y_vals = clean[var_y].values

# Step 1: Compute the actual correlation
actual_corr = np.corrcoef(x_vals, y_vals)[0, 1]
print(f'Actual correlation between {var_x} and {var_y}: r = {actual_corr:.3f}')

# Step 2: Simulate the null distribution by shuffling one variable
n_permutations = 10_000
null_corrs = np.empty(n_permutations)
for i in range(n_permutations):
    shuffled_y = np.random.permutation(y_vals)
    null_corrs[i] = np.corrcoef(x_vals, shuffled_y)[0, 1]

# Step 3: Compute p-value (two-sided)
p_value_corr_sim = np.mean(np.abs(null_corrs) >= np.abs(actual_corr))
print(f'Simulation-based p-value (two-sided): {p_value_corr_sim:.4f}')

# Plot the null distribution
plt.figure(figsize=(10, 5))
plt.hist(null_corrs, bins=50, density=True, alpha=0.7, color='seagreen',
         edgecolor='white', label='Null distribution')
plt.axvline(actual_corr, color='red', linewidth=2, linestyle='--',
            label=f'Actual r = {actual_corr:.3f}')
plt.axvline(-actual_corr, color='red', linewidth=2, linestyle='--', alpha=0.5)
plt.xlabel(f'Correlation between {var_x} and {var_y}')
plt.ylabel('Density')
plt.title(f'Permutation Test: {var_x} vs {var_y} Correlation')
plt.legend()
plt.tight_layout()
plt.show()

## Comparing to standard tests

How do our simulation-based p-values compare to the ones from classical statistical tests? Let's find out! If our simulations are working correctly, the p-values should be very close to the ones produced by `scipy.stats`.

In [ ]:
# Side-by-side comparison of simulation vs. standard tests

# --- Test 1: Sleep / Stress group comparison ---
t_stat, p_value_ttest = stats.ttest_ind(high_sleep, low_sleep)

print('=== Sleep vs. Stress (group comparison) ===')
print(f'  Simulation p-value:     {p_value_sim:.4f}')
print(f'  scipy t-test p-value:   {p_value_ttest:.4f}')
print(f'  t-statistic:            {t_stat:.3f}')
print()

# --- Test 2: Screen Time / Happiness correlation ---
r_scipy, p_value_pearson = stats.pearsonr(x_vals, y_vals)

print('=== Screen Time vs. Happiness (correlation) ===')
print(f'  Simulation p-value:     {p_value_corr_sim:.4f}')
print(f'  scipy Pearson p-value:  {p_value_pearson:.4f}')
print(f'  Pearson r:              {r_scipy:.3f}')
print()
print('The simulation-based and standard p-values should be close!')
print('Small differences are expected because simulations involve randomness.')

## Visualization

In [ ]:
# Box plot: stress levels for high-sleep vs low-sleep groups
df_plot = df[['sleep_hours', 'stress_level']].dropna().copy()
df_plot['sleep_group'] = df_plot['sleep_hours'].apply(
    lambda x: 'High sleep (>= 7 hrs)' if x >= 7 else 'Low sleep (< 7 hrs)'
)

plt.figure(figsize=(8, 6))
sns.boxplot(data=df_plot, x='sleep_group', y='stress_level', hue='sleep_group', palette='Set2', legend=False)
sns.stripplot(data=df_plot, x='sleep_group', y='stress_level',
              color='black', alpha=0.5, size=5, jitter=True)
plt.xlabel('Sleep Group')
plt.ylabel('Stress Level (1-10)')
plt.title('Stress Levels by Sleep Group')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: screen time vs happiness with regression line
plt.figure(figsize=(8, 6))
sns.regplot(data=df, x='screen_time', y='happiness',
            scatter_kws={'alpha': 0.6, 's': 60}, line_kws={'color': 'red'})
plt.xlabel('Screen Time (hours/day, outside schoolwork)')
plt.ylabel('Happiness (1-10)')
plt.title('Screen Time vs. Happiness')
plt.tight_layout()
plt.show()

## Closing notes

The simulation-based approach to statistical testing is extremely flexible. Unlike classical tests (t-tests, ANOVA, Pearson correlations, etc.), which each require specific assumptions about distributions and data types, the permutation approach works for **any** test statistic you can compute. Want to compare medians instead of means? Just change one line of code. Want to test whether the *variance* differs between groups? Same approach.

Key takeaways:

- **Every statistical test is fundamentally the same idea:** compare what you observed to what you would expect by chance.
- **Simulation lets you build that "chance" distribution directly**, without needing to derive mathematical formulas.
- **The classical tests are shortcuts** -- they give you the same answer faster by using mathematical approximations, but the logic is identical.
- **When the classical test exists, the p-values match.** This is reassuring! But the simulation approach also works in situations where no classical test exists.

Try exploring other relationships in the survey data. What hypotheses can you come up with? Can you adapt the permutation test code to test them? Experiment freely -- that is how you learn!